# Iridium-1 — TPU v5e-1 Colab builder

Choose a named model size, then optionally override the **control-core layer
count**, **superstack layer count**, and **number of superstacks**. The notebook
builds the resulting real Iridium configuration and prints exact parameter and
memory accounting before any tensor allocation.

**Runtime → Change runtime type → TPU**, then run the setup and builder cells.
`34m` and `100m` are the single-v5e training presets. `1b`, `8b`, and `25b`
are available to build and cost on this TPU notebook, but are not full-parameter
training targets for one v5e-1: full Adam state alone exceeds device memory.
This notebook refuses an unsafe training selection instead of OOMing midway.


In [ ]:
!git clone --depth 1 https://github.com/sporadicstudiosind-cloud/test.git iridium 2>/dev/null || (cd iridium && git pull)
%cd iridium
!pip -q install pyyaml

import torch
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print('XLA device:', device)
print('world size:', xm.xrt_world_size())


## 1 · Size and architecture controls

Set a preset, or leave any override as `None` to retain its preset value. The
heads are recomputed from model width, and specializations are trimmed to match
the number of superstacks, so the generated configuration remains internally
valid.


In [ ]:
# Named builds: 34m | 100m | 1b | 8b | 25b
PRESET = '34m'

# Optional architecture overrides. Use None to preserve the preset geometry.
CORE_LAYERS = None
SUPERSTACK_LAYERS = None
N_SUPERSTACKS = None

# Training controls. On one v5e-1, only 34m and 100m are enabled below.
STEPS = 100
BATCH = 4
LR = 3e-4
RUN_TRAINING = False

PRESET_TO_RUNG = {
    '34m': 'nano',
    '100m': 'nano100m',
    '1b': 'test1b',
    '8b': '8b',
    '25b': 'small',
}
assert PRESET in PRESET_TO_RUNG, f'choose one of {sorted(PRESET_TO_RUNG)}'


In [ ]:
from dataclasses import replace
from iridium.config import IridiumConfig, get_config, SPECIALIZATIONS_32

base = get_config(PRESET_TO_RUNG[PRESET])
core = replace(base.core, n_layers=CORE_LAYERS or base.core.n_layers)
n_stacks = N_SUPERSTACKS or base.stacks.n_stacks
specializations = base.stacks.specializations
if specializations and len(specializations) != n_stacks:
    specializations = SPECIALIZATIONS_32[:n_stacks] if n_stacks <= len(SPECIALIZATIONS_32) else ()
stacks = replace(
    base.stacks,
    n_layers=SUPERSTACK_LAYERS or base.stacks.n_layers,
    n_stacks=n_stacks,
    specializations=specializations,
    core_d_model=core.d_model,
)
cfg = IridiumConfig(
    name=f'{base.name}-custom', core=core, stacks=stacks,
    router=base.router, codecs=base.codecs, max_seq_len=base.max_seq_len,
    dropout=base.dropout, notes=base.notes,
)

state_gib = cfg.training_state_bytes() / 2**30
weight_gib = cfg.weight_bytes(16) / 2**30
print(cfg.report().render())
print(f'BF16 weights: {weight_gib:.1f} GiB')
print(f'full BF16 Adam state, excluding activations: {state_gib:.1f} GiB')
print(f'core layers={cfg.core.n_layers}; superstack layers={cfg.stacks.n_layers}; stacks={cfg.stacks.n_stacks}')


## 2 · TPU training gate

A v5e-1 is suitable here for the 34M and 100M single-device presets. The gate
uses the actual configured state estimate, not the preset name alone. It blocks
full-parameter training of 1B, 8B, and 25B on one TPU; these require a sharded
distributed trainer, which this repository does not implement yet.


In [ ]:
# Do not remove this gate: batch size does not reduce optimizer state.
SINGLE_V5E_TRAINABLE = {'34m', '100m'}
if RUN_TRAINING and PRESET not in SINGLE_V5E_TRAINABLE:
    raise RuntimeError(
        f'{PRESET} requires {state_gib:.1f} GiB of Adam state before activations. '
        'A single v5e-1 cannot full-train it; use a sharded distributed implementation.'
    )
if RUN_TRAINING:
    print(f'Training {PRESET} on {device} with XLA optimizer steps.')
else:
    print('Build/accounting only. Set RUN_TRAINING = True for 34m or 100m.')


## 3 · Train 34M or 100M on TPU

The corpus is the repository's existing verifiable synthetic training mixture.
It is useful for exercising this custom architecture and its routing/physics
losses; it is **not** a general internet-scale pretraining corpus.


In [ ]:
if RUN_TRAINING:
    from pathlib import Path
    from iridium.model.iridium1 import Iridium1
    from iridium.training.datasets import build_corpus, describe
    from iridium.training.losses import LossWeights
    from iridium.training.trainer import TrainConfig, Trainer

    # Keep field_rollout off until the XLA runtime's FFT support is explicitly
    # verified for this Colab image. The other task families still exercise the
    # codecs, control core, router, bridge, depth ladder, and ponder loop.
    mixture = {
        'channel_depth': 0.40,
        'channel_intervention': 0.30,
        'false_premise': 0.15,
        'scene_goal': 0.15,
    }
    train = build_corpus(12_000, seed=0, split='train', mixture=mixture)
    print(describe(train))

    model = Iridium1(cfg).to(device)
    tcfg = TrainConfig(
        steps=STEPS, batch_size=BATCH, lr=LR, seed=0,
        label=f'tpu-v5e-{PRESET}', log_every=max(STEPS // 25, 1),
        checkpoint_every=max(STEPS // 5, 1),
    )
    trainer = Trainer(model, train, tcfg, LossWeights(),
                      out_dir=Path('runs/tpu-v5e'), device=str(device))
    trainer.train()
    xm.mark_step()
    print('training complete')


## 4 · What the larger buttons mean

The builder supports 1B, 8B, and 25B so their exact custom-architecture counts
can be inspected and architecture controls can be explored. It does not pretend
a free single-core TPU can train them. A real run needs an implemented sharded
dispatcher, sharded optimizer/checkpointing, a licensed multimodal corpus, and
held-out evaluation. See `docs/training-8b.md` for the 8B boundary.
